In [1]:
import time

from sklearn.datasets import load_iris, fetch_20newsgroups, fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# 2分类估计器

In [6]:
# k近邻
"""
k-近邻预测用户签到的位置
"""

# 读取数据
data = pd.read_csv(".//data//FBlocation//train.csv")
# 查看数据前5行
print(data.head())
print(data.shape)
print(data.info)
# 处理数据:缩小数据的范围，查询数据，减少数据计算时间
data =data.query("x > 1.0 &  x < 1.25 & y > 2.5 & y < 2.75")  # x这一属性值要大于1.0且小于1.25，y这一属性值要大于2.5且要小于2.75


   row_id       x       y  accuracy    time    place_id
0       0  0.7941  9.0809        54  470702  8523065625
1       1  5.9567  4.7968        13  186555  1757726713
2       2  8.3078  7.0407        74  322648  1137537235
3       3  7.3665  2.5165        65  704587  6567393236
4       4  4.0961  1.1307        31  472130  7440663949
(29118021, 6)
<bound method DataFrame.info of             row_id       x       y  accuracy    time    place_id
0                0  0.7941  9.0809        54  470702  8523065625
1                1  5.9567  4.7968        13  186555  1757726713
2                2  8.3078  7.0407        74  322648  1137537235
3                3  7.3665  2.5165        65  704587  6567393236
4                4  4.0961  1.1307        31  472130  7440663949
...            ...     ...     ...       ...     ...         ...
29118016  29118016  6.5133  1.1435        67  399740  8671361106
29118017  29118017  5.9186  4.4134        67  125480  9077887898
29118018  29118018  2.9993  6.368

In [11]:
print(data.shape)
# 查看数据有没有空值
data.describe()  # 发现每一个的count均为17710，所以并没有缺失值

(17710, 6)


,row_id,x,y,accuracy,time,place_id
count,1.771000e+04,17710.000000,17710.000000,17710.000000,17710.000000,1.771000e+04
mean,1.450569e+07,1.122538,2.632309,82.482101,397551.263128,5.129895e+09
std,8.353805e+06,0.077086,0.070144,113.613227,234601.097883,2.357399e+09
min,6.000000e+02,1.000100,2.500100,1.000000,119.000000,1.012024e+09
25%,7.327816e+06,1.049200,2.573800,25.000000,174069.750000,3.312464e+09
50%,1.443071e+07,1.123300,2.642300,62.000000,403387.500000,5.261906e+09
75%,2.163463e+07,1.190500,2.687800,75.000000,602111.750000,6.766325e+09
max,2.911215e+07,1.249900,2.749900,1004.000000,786218.000000,9.980711e+09


In [13]:
# 处理时间的时间,unit = s是秒的意思，将数据中所给的秒的数据转换成日期的格式
time_value =pd.to_datetime(data['time'],unit='s')
print(time_value[:10])

600    1970-01-01 18:09:40
957    1970-01-10 02:11:10
4345   1970-01-05 15:08:02
4735   1970-01-06 23:03:03
5580   1970-01-09 11:26:50
6090   1970-01-02 16:25:07
6234   1970-01-04 15:52:57
6350   1970-01-01 10:13:36
7468   1970-01-09 15:26:06
8478   1970-01-08 23:52:02
Name: time, dtype: datetime64[ns]


In [14]:
# pd.DatetimeIndex() 用来把 日期字符串或时间数据 转换成 pandas 的时间索引对象。
# 把日期格式转换成字典格式，并将时间这一列变成索引
time_value = pd.DatetimeIndex(time_value)
print('-'*50)
print(time_value[:5])

--------------------------------------------------
DatetimeIndex(['1970-01-01 18:09:40', '1970-01-10 02:11:10',
               '1970-01-05 15:08:02', '1970-01-06 23:03:03',
               '1970-01-09 11:26:50'],
              dtype='datetime64[ns]', name='time', freq=None)


In [19]:
data.insert(data.shape[1], 'day', time_value.day) #data.shape[1]是代表插入到最后的意思,一个月的哪一天
data.insert(data.shape[1], 'hour', time_value.hour)#是否去一个地方打卡，早上，中午，晚上是有影响的
data.insert(data.shape[1], 'weekday', time_value.weekday) #0代表周一，6代表周日，星期几
# 把时间戳特征删除
data = data.drop(['time'], axis=1)
data.head()

,row_id,x,y,accuracy,time,place_id,day,hour,weekday
600,600,1.2214,2.7023,17,65380,6683426742,1,18,3
957,957,1.1832,2.6891,58,785470,6683426742,10,2,5
4345,4345,1.1935,2.6550,11,400082,6889790653,5,15,0
4735,4735,1.1452,2.6074,49,514983,6822359752,6,23,1
5580,5580,1.0089,2.7287,19,732410,1527921905,9,11,4


In [21]:
# # 把签到数量少于n个目标位置删除，place_id是标签，即目标值
place_count = data.groupby('place_id').count()
"""place_count 返回的是一个dataFrame，place_count的属性与原来的data属性完全相同，里面的内容改成对应每个place_id的属性值所出现分次数
"""
print(type(place_count))
place_count

<class 'pandas.core.frame.DataFrame'>


,row_id,x,y,accuracy,time,day,hour,weekday
place_id,,,,,,,,
1012023972,1,1,1,1,1,1,1,1
1057182134,1,1,1,1,1,1,1,1
1059958036,3,3,3,3,3,3,3,3
1085266789,1,1,1,1,1,1,1,1
1097200869,1044,1044,1044,1044,1044,1044,1044,1044
...,...,...,...,...,...,...,...,...
9904182060,1,1,1,1,1,1,1,1
9915093501,1,1,1,1,1,1,1,1
9946198589,1,1,1,1,1,1,1,1


In [22]:
place_count['x'].describe() #查看各个地点打卡点的打卡的次数

count     805.000000
mean       22.000000
std        88.955632
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max      1044.000000
Name: x, dtype: float64

In [23]:
# 删去打卡次数小于4次的地点，减少噪音对数据产生的影响
tf = place_count[place_count.row_id > 3].reset_index()  # reset_index()重新让其编号,生成0，1，2...变成索引，将原来的索引变成普通索引
tf  #剩余的签到地点

,place_id,row_id,x,y,accuracy,time,day,hour,weekday
0,1097200869,1044,1044,1044,1044,1044,1044,1044,1044
1,1228935308,120,120,120,120,120,120,120,120
2,1267801529,58,58,58,58,58,58,58,58
3,1278040507,15,15,15,15,15,15,15,15
4,1285051622,21,21,21,21,21,21,21,21
...,...,...,...,...,...,...,...,...,...
234,9741307878,5,5,5,5,5,5,5,5
235,9753855529,21,21,21,21,21,21,21,21
236,9806043737,6,6,6,6,6,6,6,6
237,9809476069,23,23,23,23,23,23,23,23


In [24]:
# 根据设定的地点目标值，对原本的样本进行过滤，将签到少于四次的样本进行删除
#isin可以过滤某一列要在一组值，用于判断某一列的值是否在指定的集合中
data = data[data['place_id'].isin(tf.place_id)]
data.shape

(16918, 9)

In [25]:
# # 取出数据当中的特征值和目标值,y就为我们需要得到的结果y
y = data['place_id']
# 删除目标值，保留特征值（删除place_id这一列）x
x = data.drop(['place_id'], axis=1)
# 删除无用的特征值，row_id是索引,这就是噪音
x = x.drop(['row_id'], axis=1)
print(x.shape)
print(x.columns)  # 查看现在x中的属性值有哪些,列索引

(16918, 7)
Index(['x', 'y', 'accuracy', 'time', 'day', 'hour', 'weekday'], dtype='object')


# 上面预处理完成

In [28]:
# 现在进行开始训练，训练出f(x)
# 1.分割数据集、
x_train,x_test,y_train,y_test =train_test_split(x,y,test_size=0.25,random_state=1)

# 2.进行标准化并进行训练
std =StandardScaler()
x_train = std.fit_transform(x_train)
print(std.mean_)
print(std.var_)
print('-'*50)
x_test =std.transform(x_test)
print(std.mean_)
print(std.var_)

[1.12295735e+00 2.63237278e+00 8.13493852e+01 3.97291479e+05
 5.10064628e+00 1.14429382e+01 3.10135561e+00]
[5.98489138e-03 4.86857391e-03 1.19597480e+04 5.51124483e+10
 7.32837915e+00 4.83742660e+01 2.81838404e+00]
--------------------------------------------------
[1.12295735e+00 2.63237278e+00 8.13493852e+01 3.97291479e+05
 5.10064628e+00 1.14429382e+01 3.10135561e+00]
[5.98489138e-03 4.86857391e-03 1.19597480e+04 5.51124483e+10
 7.32837915e+00 4.83742660e+01 2.81838404e+00]


In [33]:
# 进行knn算法,能自己进行修改的参数叫做超参数
knn =KNeighborsClassifier(n_neighbors = 7)  # 设置邻居数量，来调整结果好坏
# fit,predict,score，训练，knn的fit是不训练的，只是把训练集的特征值和目标值放入到内存中
knn.fit(x_train, y_train)
# 得到预测的结果
y_predict = knn.predict(x_test)
print("预测的目标签到位置为：", y_predict[0:10])
#得出预测的准确率，是评估指标
print(f"预测的准确率:{knn.score(x_test, y_test) * 100}%")

预测的目标签到位置为： [1913341282 1097200869 6097504486 9632980559 6424972551 4022692381
 8048985799 6683426742 1435128522 3312463746]
预测的准确率:48.25059101654846%


# 调整超参的方法，网格搜索，可以找出最优的超参数


In [34]:
# 构造一些参数的值进行搜素，uniform有权重，distance直接在按照是距离
param = {"n_neighbors": [3, 5, 10, 12, 15],'weights':['uniform', 'distance']}

# 进行网格搜索，cv = 3,是3折交叉验证，1折验证，2折训练
gc =GridSearchCV(knn,param_grid=param,cv=3)

gc.fit(x_train,y_train)
print("在测试集上准确率：", gc.score(x_test, y_test))

print("在交叉验证当中最好的结果：", gc.best_score_) #最好的结果

print("选择最好的模型是：", gc.best_estimator_) #最好的模型,告诉你用了哪些参数

print("每个超参数每次交叉验证的结果：")
gc.cv_results_  # cv_results_其中包含了一组参数中交叉验证中的所有结果


C:\Users\asus\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


在测试集上准确率： 0.49929078014184397
在交叉验证当中最好的结果： 0.4805326127282427
选择最好的模型是： KNeighborsClassifier(n_neighbors=10, weights='distance')
每个超参数每次交叉验证的结果：


{'mean_fit_time': array([0.00785764, 0.00781059, 0.01051688, 0.00885955, 0.00830452,
        0.00874813, 0.00780217, 0.00779176, 0.00778325, 0.00777523]),
 'std_fit_time': array([5.95201317e-05, 2.28459246e-05, 1.87851817e-03, 7.56029922e-04,
        7.50347987e-04, 1.36064888e-03, 4.20380629e-06, 4.77234357e-06,
        3.13810137e-05, 4.20141674e-05]),
 'mean_score_time': array([0.12776677, 0.10800695, 0.15322471, 0.05627163, 0.142893  ,
        0.07321572, 0.14767281, 0.08435933, 0.15834022, 0.09045712]),
 'std_score_time': array([7.78161486e-03, 8.66192424e-02, 1.44426437e-02, 9.30548118e-05,
        1.72405577e-03, 2.49709402e-03, 5.86877323e-03, 7.51177782e-03,
        9.02330782e-04, 2.56831064e-03]),
 'param_n_neighbors': masked_array(data=[3, 3, 5, 5, 10, 10, 12, 12, 15, 15],
              mask=[False, False, False, False, False, False, False, False,
                    False, False],
        fill_value='?',
             dtype=object),
 'param_weights': masked_array(data=['uni

In [35]:
"""
朴素贝叶斯进行文本分类
"""
news = fetch_20newsgroups(subset='all',data_home='data')

print(len(news.data))  #样本数，包含的特征
print('-'*50)
print(news.data[0]) #第一个样本 特征
print('-'*50)
print(news.target) #标签
print(np.unique(news.target)) #标签的类别
print(news.target_names) #标签的名字

18846
--------------------------------------------------
From: Mamatha Devineni Ratnam <mr47+@andrew.cmu.edu>
Subject: Pens fans reactions
Organization: Post Office, Carnegie Mellon, Pittsburgh, PA
Lines: 12
NNTP-Posting-Host: po4.andrew.cmu.edu



I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse than I thought. Jagr just showed you why
he is much better than his regular season stats. He is also a lot
fo fun to watch in the playoffs. Bowman should let JAgr have a lot of
fun in the next couple of games since the Pens are going to beat the pulp out of Jersey anyway. I was very disappointed not to see the Islanders lose the final
regular season game.          PENS RULE!!!


----------------------------------------

In [36]:
# 进行数据分割
x_train,x_test,y_train,y_test =train_test_split(news.data,news.target,test_size=0.25,random_state=1)

# 对数据集进行特征提取,将字符串类型转换成数值类型
tf =TfidfVectorizer()

# 以训练集当中的词的列表进行每篇文章重要性统计
x_train = tf.fit_transform(x_train)

#针对特征内容，可以自行打印，下面的打印可以得到特征数目，总计有15万特征
print(len(tf.get_feature_names_out()))

153196


In [37]:
import time
# 设置拉普拉斯平滑参数，防止某个词在某个类别中出现的次数为0
mlt =MultinomialNB(alpha=1.0)
start = time.time()
mlt.fit(x_train,y_train)  # mlt就是已经生成过的估计器，就是一个工具
end = time.time()
print(end-start)

0.1627497673034668


In [39]:
x_transform_test =tf.transform(x_test)
print(len(tf.get_feature_names_out()))  # 查看特征数目

153196


In [40]:
start = time.time()
y_predict =mlt.predict(x_transform_test)
print("预测的前面10篇文章类别为：", y_predict[0:10])

# 得出准确率,这个是很难提高准确率，为什么呢？
print("准确率为：", mlt.score(x_transform_test, y_test))
end=time.time()
end-start

预测的前面10篇文章类别为： [16 19 18  1  9 15  1  2 16 13]
准确率为： 0.8518675721561969


0.06049084663391113

In [41]:
# 目前这个场景我们不需要召回率，support是真实的为那个类别的有多少个样本
print(classification_report(y_test, y_predict,target_names=news.target_names))

                          precision    recall  f1-score   support

             alt.atheism       0.91      0.77      0.83       199
           comp.graphics       0.83      0.79      0.81       242
 comp.os.ms-windows.misc       0.89      0.83      0.86       263
comp.sys.ibm.pc.hardware       0.80      0.83      0.81       262
   comp.sys.mac.hardware       0.90      0.88      0.89       234
          comp.windows.x       0.92      0.85      0.88       230
            misc.forsale       0.96      0.67      0.79       257
               rec.autos       0.90      0.87      0.88       265
         rec.motorcycles       0.90      0.95      0.92       251
      rec.sport.baseball       0.89      0.96      0.93       226
        rec.sport.hockey       0.95      0.98      0.96       262
               sci.crypt       0.76      0.97      0.85       257
         sci.electronics       0.84      0.80      0.82       229
                 sci.med       0.97      0.86      0.91       249
         

In [42]:
y_test1 = np.where(y_test == 0, 1, 0)  # 为类别1也就是alt.atheism的个数，三目运算符
print(y_test1.sum()) #实际为真的个数

199


In [43]:
y_predict1 = np.where(y_predict == 0, 1, 0) # 预测为类别1的个数
print(y_predict1.sum())# 预测为真的个数

168


In [44]:
(y_test1*y_predict1).sum()  # 预测正确且为类别1的个数

153

In [45]:
# 把0-19总计20个分类，变为0和1
# 5是可以改为0到19的
y_test1 = np.where(y_test == 5, 1, 0)
print(y_test1.sum()) #label为5的样本数
y_predict1 = np.where(y_predict == 5, 1, 0)
print(y_predict1.sum())
# roc_auc_score的y_test只能是二分类,针对多分类如何计算AUC，roc纵轴是召回率，横轴是FPR
print("AUC指标：", roc_auc_score(y_test1, y_predict1))

230
214
AUC指标： 0.924078924393225


# 3 决策树

In [46]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

In [47]:
"""
决策树对泰坦尼克号进行预测生死
:return: None
"""
# 获取数据
titan = pd.read_csv("./data/titanic.txt")
titan.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1313 entries, 0 to 1312
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   row.names  1313 non-null   int64  
 1   pclass     1313 non-null   object 
 2   survived   1313 non-null   int64  
 3   name       1313 non-null   object 
 4   age        633 non-null    float64
 5   embarked   821 non-null    object 
 6   home.dest  754 non-null    object 
 7   room       77 non-null     object 
 8   ticket     69 non-null     object 
 9   boat       347 non-null    object 
 10  sex        1313 non-null   object 
dtypes: float64(1), int64(2), object(8)
memory usage: 113.0+ KB


In [48]:
# 处理数据，找出特征值和目标值
x = titan[['pclass', 'age', 'sex']]

y = titan['survived']
print(x.info())  # 用来判断是否有空值
x.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1313 entries, 0 to 1312
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   pclass  1313 non-null   object 
 1   age     633 non-null    float64
 2   sex     1313 non-null   object 
dtypes: float64(1), object(2)
memory usage: 30.9+ KB
None


,pclass,age,sex
count,1313,633.000000,1313
unique,3,NaN,2
top,3rd,NaN,male
freq,711,NaN,850
mean,NaN,31.194181,NaN
std,NaN,14.747525,NaN
min,NaN,0.166700,NaN
25%,NaN,21.000000,NaN
50%,NaN,30.000000,NaN
75%,NaN,41.000000,NaN


In [51]:
# 一定要进行缺失值处理,填为均值（将确实的年龄数据用年龄的均值代替）
mean=x['age'].mean()
x.loc[:,'age']=x.loc[:,'age'].fillna(mean)
x.info() # 再次查看是否有缺失值

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1313 entries, 0 to 1312
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   pclass  1313 non-null   object 
 1   age     1313 non-null   float64
 2   sex     1313 non-null   object 
dtypes: float64(1), object(2)
memory usage: 30.9+ KB


In [52]:
# 分割数据集到训练集合测试集
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=4)
print(x_train.head())

    pclass        age     sex
598    2nd  30.000000    male
246    1st  62.000000    male
905    3rd  31.194181  female
300    1st  31.194181  female
509    2nd  64.000000    male


In [53]:
#女性中存活的情况对比男性中存活情况
z=x_train.copy() #z是为了把特征和目标存储到一起
z['survived'] = y_train #把目标值存储到z中
z[z['sex'] == 'female']['survived'].value_counts() #男性中存活的情况

survived
1    230
0    111
Name: count, dtype: int64

In [55]:
z[z['sex'] == 'male']['survived'].value_counts()

survived
0    539
1    104
Name: count, dtype: int64

In [56]:
#把df变为字典，样本变为一个一个的字典，字典中列名变为键，orient是指定数据的组织方式，默认为list列表
x_train.to_dict(orient="records")

[{'pclass': '2nd', 'age': 30.0, 'sex': 'male'},
 {'pclass': '1st', 'age': 62.0, 'sex': 'male'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'female'},
 {'pclass': '1st', 'age': 31.19418104265403, 'sex': 'female'},
 {'pclass': '2nd', 'age': 64.0, 'sex': 'male'},
 {'pclass': '1st', 'age': 31.19418104265403, 'sex': 'female'},
 {'pclass': '3rd', 'age': 24.0, 'sex': 'female'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'male'},
 {'pclass': '2nd', 'age': 31.19418104265403, 'sex': 'male'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'male'},
 {'pclass': '3rd', 'age': 21.0, 'sex': 'male'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'male'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'male'},
 {'pclass': '2nd', 'age': 23.0, 'sex': 'female'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'male'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'female'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'female'},
 {'pclass': '1st', 'age': 4

In [57]:
# 进行处理（特征工程）特征-》类别-》one_hot编码,sparse不是稀疏矩阵三元组方式的来存，而是直接用矩阵来存储稀疏矩阵
dict = DictVectorizer(sparse=False)

# 这一步是对字典进行特征抽取,to_dict可以把df变为字典，records代表列名变为键
x_train = dict.fit_transform(x_train.to_dict(orient="records"))
print(type(x_train))
print(dict.get_feature_names_out())
print('-' * 50)
x_test = dict.transform(x_test.to_dict(orient="records"))
print(x_train)

<class 'numpy.ndarray'>
['age' 'pclass=1st' 'pclass=2nd' 'pclass=3rd' 'sex=female' 'sex=male']
--------------------------------------------------
[[30.          0.          1.          0.          0.          1.        ]
 [62.          1.          0.          0.          0.          1.        ]
 [31.19418104  0.          0.          1.          1.          0.        ]
 ...
 [34.          0.          1.          0.          0.          1.        ]
 [46.          1.          0.          0.          0.          1.        ]
 [31.19418104  0.          0.          1.          0.          1.        ]]


In [58]:
#树过于复杂，就会产生过拟合
dec = DecisionTreeClassifier()  # dec这种叫做估计器

#训练
dec.fit(x_train, y_train)

# 预测准确率
print("预测的准确率：", dec.score(x_test, y_test))

# 导出决策树的结构
export_graphviz(dec, out_file="tree.dot",
                feature_names=['age', 'pclass=1st', 'pclass=2nd', 'pclass=3rd', 'female', 'male'])


预测的准确率： 0.8085106382978723


# 对决策树进行参数调优

In [61]:
# 数据分割
x_train, x_test, y_train, y_test =train_test_split(x,y,test_size=0.25,random_state=4)
# 进行特征工程
dict = DictVectorizer(sparse=False)
# 进行特征提取
x_train = dict.fit_transform(x_train.to_dict(orient='records'))
x_test = dict.fit_transform(x_test.to_dict(orient='records'))

# 生成估计器
# 用决策树进行预测，修改max_depth为10，发现提升了,min_impurity_decrease带来的增益要大于0.01才会进行划分，因此可以用来去除一些决定性非常小的杂质
dec = DecisionTreeClassifier(max_depth=7,min_impurity_decrease=0.01,min_samples_split=20)

# 进行训练
dec.fit(x_train,y_train)

# 预测的准确率
print("预测的准确率：",dec.score(x_test,y_test))

# 导出决策树的结构
export_graphviz(dec, out_file="tree1.dot",
                feature_names=dict.get_feature_names_out())

预测的准确率： 0.8206686930091185


In [62]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=4)
# 进行处理（特征工程）特征-》类别-》one_hot编码
dict = DictVectorizer(sparse=False)

# 这一步是对字典进行特征抽取
x_train = dict.fit_transform(x_train.to_dict(orient="records"))
x_test = dict.transform(x_test.to_dict(orient="records"))

# 随机森林进行预测 （超参数调优），n_jobs充分利用多核的一个参数
rf = RandomForestClassifier(n_jobs=-1)
# 120, 200, 300, 500, 800, 1200,n_estimators森林中决策树的数目，也就是分类器的数目
# max_samples  是最大样本数
#bagging类型
param = {"n_estimators": [1500,2000, 5000], "max_depth": [2, 3, 5, 8, 15, 25]}

# 网格搜索与交叉验证
gc = GridSearchCV(rf, param_grid=param, cv=3)

gc.fit(x_train, y_train)

print("准确率：", gc.score(x_test, y_test))

print("查看选择的参数模型：", gc.best_params_)

print("选择最好的模型是：", gc.best_estimator_)

准确率： 0.8328267477203647
查看选择的参数模型： {'max_depth': 3, 'n_estimators': 1500}
选择最好的模型是： RandomForestClassifier(max_depth=3, n_estimators=1500, n_jobs=-1)


In [63]:
print("每个超参数每次交叉验证的结果：", gc.cv_results_)

每个超参数每次交叉验证的结果： {'mean_fit_time': array([1.59617742, 1.88420208, 4.70957605, 1.43756032, 2.08386   ,
       4.74934101, 1.44360344, 1.91289949, 4.76227593, 1.44744237,
       1.92085393, 4.77110227, 1.44409649, 1.9253935 , 4.77010894,
       1.44699987, 1.93808015, 4.79193497]), 'std_fit_time': array([0.21872583, 0.00539527, 0.00690663, 0.00448971, 0.26483935,
       0.02368885, 0.00626637, 0.00416435, 0.00667294, 0.0062572 ,
       0.00877001, 0.00679889, 0.004681  , 0.00476398, 0.0282314 ,
       0.00581392, 0.01224498, 0.00890813]), 'mean_score_time': array([0.14800088, 0.18828503, 0.4534572 , 0.17204634, 0.20619122,
       0.52514299, 0.16015752, 0.20769827, 0.50772039, 0.16023119,
       0.21548017, 0.51007589, 0.16495458, 0.21467026, 0.51399032,
       0.16491477, 0.21488921, 0.51101589]), 'std_score_time': array([0.00379251, 0.00457961, 0.01787022, 0.01780061, 0.00496245,
       0.01835874, 0.00108192, 0.00605162, 0.00636277, 0.00056775,
       0.00069094, 0.00941418, 0.00629672